<a href="https://www.kaggle.com/code/likithagedipudi/scientific-image-forgery-detection?scriptVersionId=301326597" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Scientific Image Forgery Detection — PyTorch baseline

In [ ]:
 !pip -q install -U albumentations==1.4.20 opencv-python-headless==4.10.0.84 segmentation-models-pytorch==0.3.4


In [ ]:
import os
import gc
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image

# Optional deps
try:
    import cv2
except Exception:
    cv2 = None

try:
    import albumentations as A
except Exception:
    A = None

try:
    import segmentation_models_pytorch as smp
except Exception:
    smp = None

try:
    import torchvision
    from torchvision.models.segmentation import deeplabv3_resnet50
    try:
        from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights
    except Exception:
        DeepLabV3_ResNet50_Weights = None
except Exception:
    torchvision = None
    deeplabv3_resnet50 = None
    DeepLabV3_ResNet50_Weights = None


In [ ]:
from contextlib import nullcontext

def get_device() -> str:
    if torch.cuda.is_available():
        return 'cuda'
    # Apple Silicon
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

@dataclass
class CFG:
    seed: int = 42
    device: str = get_device()

    # training
    epochs: int = 10
    lr: float = 3e-4
    weight_decay: float = 1e-4
    batch_size: int = 8
    # In notebooks on macOS, num_workers>0 often breaks due to multiprocessing pickling.
    num_workers: int = 0

    # preprocessing
    img_size: int = 768

    # inference
    thr: float = 0.5

    # CV
    n_splits: int = 5
    fold: int = 0

    # speed/stability
    use_amp: bool = True


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(CFG.seed)

# Kaggle vs local notebook defaults
RUNNING_ON_KAGGLE = os.environ.get('KAGGLE_URL_BASE') is not None
# Kaggle supports DataLoader workers; macOS Jupyter often does not (spawn/pickling).
CFG.num_workers = 2 if RUNNING_ON_KAGGLE else 0

AMP_ENABLED = CFG.use_amp and (CFG.device == 'cuda')
PIN_MEMORY = (CFG.device == 'cuda')
print('device:', CFG.device, 'AMP:', AMP_ENABLED, 'pin_memory:', PIN_MEMORY, 'num_workers:', CFG.num_workers, 'kaggle:', RUNNING_ON_KAGGLE)


In [ ]:
# Paths (Kaggle + local fallback)
CANDIDATES = [
    Path('/kaggle/input/recodai-luc-scientific-image-forgery-detection'),
    Path('.'),
]
DATA_DIR = None
for p in CANDIDATES:
    if (p / 'train_images').exists() and (p / 'test_images').exists():
        DATA_DIR = p
        break
assert DATA_DIR is not None, 'Could not find dataset directory.'

TRAIN_AUTH_DIR = DATA_DIR / 'train_images' / 'authentic'
TRAIN_FORG_DIR = DATA_DIR / 'train_images' / 'forged'
TRAIN_MASK_DIR = DATA_DIR / 'train_masks'

SUPP_IMG_DIR = DATA_DIR / 'supplemental_images'
SUPP_MASK_DIR = DATA_DIR / 'supplemental_masks'

TEST_DIR = DATA_DIR / 'test_images'

print('DATA_DIR:', DATA_DIR)
print('train_auth:', TRAIN_AUTH_DIR.exists(), 'train_forg:', TRAIN_FORG_DIR.exists(), 'test:', TEST_DIR.exists())


In [ ]:
# RLE (OFFICIAL STYLE)
# This competition encodes INSTANCE masks: a single string contains one or more JSON arrays separated by ';'.
# For a single predicted mask, pass [mask] into rle_encode([...]).

import json

try:
    import numba
except Exception:
    numba = None

def _rle_encode_py(x: np.ndarray, fg_val: int = 1) -> list[int]:
    # Matches the official behavior: find foreground indices in column-major order.
    dots = np.where(x.T.flatten() == fg_val)[0]
    run_lengths: list[int] = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((int(b) + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return run_lengths

if numba is not None:
    @numba.jit(nopython=True)
    def _rle_encode_jit(x: np.ndarray, fg_val: int = 1):
        dots = np.where(x.T.flatten() == fg_val)[0]
        run_lengths = []
        prev = -2
        for b in dots:
            if b > prev + 1:
                run_lengths.extend((b + 1, 0))
            run_lengths[-1] += 1
            prev = b
        return run_lengths

def rle_encode(masks: list[np.ndarray], fg_val: int = 1) -> str:
    encs = []
    for x in masks:
        x = (x == fg_val).astype(np.uint8)
        if numba is not None:
            enc = _rle_encode_jit(x, fg_val)
        else:
            enc = _rle_encode_py(x, fg_val)
        encs.append(json.dumps(enc))
    return ';'.join(encs)


In [ ]:
def build_df() -> pd.DataFrame:
    rows = []

    # authentic
    for p in sorted(TRAIN_AUTH_DIR.glob('*.png')):
        cid = int(p.stem)
        rows.append({'case_id': cid, 'img_path': str(p), 'mask_path': None, 'is_forged': 0})

    # forged
    for p in sorted(TRAIN_FORG_DIR.glob('*.png')):
        cid = int(p.stem)
        m = TRAIN_MASK_DIR / f'{cid}.npy'
        rows.append({'case_id': cid, 'img_path': str(p), 'mask_path': str(m) if m.exists() else None, 'is_forged': 1})

    # supplemental (treat mask existence as forged)
    if SUPP_IMG_DIR.exists():
        for p in sorted(SUPP_IMG_DIR.glob('*.png')):
            cid = int(p.stem)
            m = SUPP_MASK_DIR / f'{cid}.npy'
            rows.append({'case_id': cid, 'img_path': str(p), 'mask_path': str(m) if m.exists() else None, 'is_forged': 1 if m.exists() else 0})

    df = pd.DataFrame(rows).drop_duplicates(subset=['case_id']).reset_index(drop=True)
    return df

df = build_df()
display(df.head())
print('df:', df.shape)
print(df['is_forged'].value_counts(dropna=False))


In [ ]:
def read_image_rgb(path: str) -> np.ndarray:
    if cv2 is not None:
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img
    return np.array(Image.open(path).convert('RGB'))

def load_union_mask(mask_path, h: int, w: int) -> np.ndarray:
    if mask_path is None:
        return np.zeros((h, w), dtype=np.uint8)
    m = np.load(mask_path)
    if m.ndim == 2:
        mm = (m > 0).astype(np.uint8)
    else:
        mm = (m > 0).any(axis=0).astype(np.uint8)
    return mm

def resize_longest_and_pad(img: np.ndarray, mask, out_size: int):
    orig_h, orig_w = img.shape[:2]
    scale = out_size / max(orig_h, orig_w)
    new_h = int(round(orig_h * scale))
    new_w = int(round(orig_w * scale))

    if cv2 is not None:
        img_rs = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        mask_rs = None
        if mask is not None:
            mask_rs = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    else:
        img_rs = np.array(Image.fromarray(img).resize((new_w, new_h), resample=Image.BILINEAR))
        mask_rs = None
        if mask is not None:
            mask_rs = np.array(Image.fromarray(mask).resize((new_w, new_h), resample=Image.NEAREST))

    pad_h = out_size - new_h
    pad_w = out_size - new_w
    img_pad = np.pad(img_rs, ((0, pad_h), (0, pad_w), (0, 0)), mode='constant', constant_values=0)

    mask_pad = None
    if mask_rs is not None:
        mask_pad = np.pad(mask_rs, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0)

    meta = {
        'scale': scale,
        'new_h': new_h,
        'new_w': new_w,
        'pad_h': pad_h,
        'pad_w': pad_w,
        'orig_h': orig_h,
        'orig_w': orig_w,
    }
    return img_pad, mask_pad, meta

def unpad_and_resize_back(pred_pad: np.ndarray, meta: dict) -> np.ndarray:
    new_h, new_w = meta['new_h'], meta['new_w']
    orig_h, orig_w = meta['orig_h'], meta['orig_w']
    pred_crop = pred_pad[:new_h, :new_w]
    if cv2 is not None:
        pred_back = cv2.resize(pred_crop, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    else:
        pred_back = np.array(Image.fromarray(pred_crop).resize((orig_w, orig_h), resample=Image.NEAREST))
    return pred_back.astype(np.uint8)

def normalize_img(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    return (img - mean) / std


In [ ]:
def get_train_aug():
    if A is None:
        return None
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(p=0.2),
        A.OneOf([A.MotionBlur(p=1.0), A.GaussianBlur(p=1.0)], p=0.2),
    ])

class ForgeryDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.aug = get_train_aug() if train else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = read_image_rgb(row.img_path)
        h, w = img.shape[:2]
        mask = load_union_mask(row.mask_path, h, w)

        if self.aug is not None:
            out = self.aug(image=img, mask=mask)
            img, mask = out['image'], out['mask']

        img_pad, mask_pad, _ = resize_longest_and_pad(img, mask, CFG.img_size)
        img_pad = normalize_img(img_pad)

        img_t = torch.from_numpy(img_pad).permute(2, 0, 1).float()
        mask_t = torch.from_numpy(mask_pad).unsqueeze(0).float()
        return img_t, mask_t


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2 * (probs * targets).sum(dim=(2,3))
        den = (probs + targets).sum(dim=(2,3)).clamp_min(self.eps)
        dice = num / den
        return 1 - dice.mean()

class _TorchvisionDeepLabV3(nn.Module):
    def __init__(self):
        super().__init__()
        if deeplabv3_resnet50 is None:
            raise RuntimeError('torchvision deeplabv3_resnet50 is not available')
        weights = None
        if DeepLabV3_ResNet50_Weights is not None:
            weights = DeepLabV3_ResNet50_Weights.DEFAULT
        m = deeplabv3_resnet50(weights=weights)
        if hasattr(m, 'classifier') and isinstance(m.classifier, nn.Sequential):
            m.classifier[-1] = nn.Conv2d(m.classifier[-1].in_channels, 1, kernel_size=1)
        else:
            m.classifier = nn.Conv2d(256, 1, kernel_size=1)
        if hasattr(m, 'aux_classifier'):
            m.aux_classifier = None
        self.model = m

    def forward(self, x):
        out = self.model(x)
        if isinstance(out, dict):
            return out['out']
        return out

def build_model() -> nn.Module:
    if smp is not None:
        return smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=1)
    if torchvision is not None and deeplabv3_resnet50 is not None:
        return _TorchvisionDeepLabV3()
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(16, 1, 1),
    )

def dice_coef_from_logits(logits: torch.Tensor, targets: torch.Tensor, thr: float = 0.5) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > thr).float()
    inter = (preds * targets).sum(dim=(2,3))
    den = (preds + targets).sum(dim=(2,3)).clamp_min(1.0)
    return (2 * inter / den).mean().item()


In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
df['fold'] = -1
for fold, (_, va_idx) in enumerate(skf.split(df, df['is_forged'])):
    df.loc[va_idx, 'fold'] = fold

train_df = df[df['fold'] != CFG.fold].reset_index(drop=True)
valid_df = df[df['fold'] == CFG.fold].reset_index(drop=True)
print('train:', train_df.shape, 'valid:', valid_df.shape)
print('valid forged rate:', valid_df['is_forged'].mean())


In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, bce, dice_loss):
    model.train()
    total_loss = 0.0
    for imgs, masks in loader:
        imgs = imgs.to(CFG.device, non_blocking=True)
        masks = masks.to(CFG.device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        autocast_ctx = torch.amp.autocast('cuda') if AMP_ENABLED else nullcontext()
        with autocast_ctx:
            logits = model(imgs)
            loss = bce(logits, masks) + dice_loss(logits, masks)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def valid_one_epoch(model, loader, bce, dice_loss):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    for imgs, masks in loader:
        imgs = imgs.to(CFG.device, non_blocking=True)
        masks = masks.to(CFG.device, non_blocking=True)
        logits = model(imgs)
        loss = bce(logits, masks) + dice_loss(logits, masks)
        total_loss += loss.item() * imgs.size(0)
        total_dice += dice_coef_from_logits(logits, masks, thr=0.5) * imgs.size(0)
    return total_loss / len(loader.dataset), total_dice / len(loader.dataset)


In [ ]:
train_ds = ForgeryDataset(train_df, train=True)
valid_ds = ForgeryDataset(valid_df, train=False)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=PIN_MEMORY, drop_last=True)
valid_loader = DataLoader(valid_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=PIN_MEMORY)

model = build_model().to(CFG.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
scaler = torch.amp.GradScaler('cuda') if AMP_ENABLED else None

bce = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()

best_dice = -1.0
best_path = 'best_model.pth'

for epoch in range(CFG.epochs):
    tr_loss = train_one_epoch(model, train_loader, optimizer, scaler, bce, dice_loss)
    va_loss, va_dice = valid_one_epoch(model, valid_loader, bce, dice_loss)
    scheduler.step()
    print(f'Epoch {epoch+1}/{CFG.epochs} | train {tr_loss:.4f} | valid {va_loss:.4f} | dice {va_dice:.4f}')
    if va_dice > best_dice:
        best_dice = va_dice
        torch.save(model.state_dict(), best_path)

print('best_dice:', best_dice)


In [ ]:
@torch.no_grad()
def predict_one(model, img_path: str) -> np.ndarray:
    img = read_image_rgb(img_path)
    img_pad, _, meta = resize_longest_and_pad(img, mask=None, out_size=CFG.img_size)
    img_pad = normalize_img(img_pad)

    x = torch.from_numpy(img_pad).permute(2, 0, 1).float().unsqueeze(0).to(CFG.device)
    logits = model(x)[0, 0]
    prob = torch.sigmoid(logits).float().cpu().numpy()
    pred_pad = (prob > CFG.thr).astype(np.uint8)

    pred = unpad_and_resize_back(pred_pad, meta)
    return pred

model.load_state_dict(torch.load(best_path, map_location=CFG.device))
model.eval()

test_paths = sorted(TEST_DIR.glob('*.png'))
print('num test:', len(test_paths))

rows = []
for p in test_paths:
    case_id = int(p.stem)
    pred = predict_one(model, str(p))
    if pred.sum() == 0:
        ann = 'authentic'
    else:
        ann = rle_encode([pred])
    rows.append({'case_id': case_id, 'annotation': ann})

from pathlib import Path

sub = pd.DataFrame(rows).sort_values('case_id').reset_index(drop=True)
out_csv = '/kaggle/working/submission.csv' if RUNNING_ON_KAGGLE else 'submission.csv'
sub.to_csv(out_csv, index=False)
display(sub.head())
p = Path(out_csv)
print('wrote', out_csv, 'exists:', p.exists(), 'size:', p.stat().st_size if p.exists() else None)
assert p.exists(), 'submission.csv was not created'

# cleanup
try:
    del rows
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
